In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from model import NSConfig, NsModel, CUR_DIR

In [2]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(f"{CUR_DIR}/model/tokenizer_sft.json")

In [4]:
def apply_chat_template(user_prompt: str):
    system_prompt = "You are a helpful assistant."
    user_text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n"
    return user_text

In [5]:
cfg = NSConfig()

device = "cuda:1"

cfg.vocab_size += 2
model = NsModel(cfg)
model.to(device)

model.load_state_dict(
    torch.load(f"{CUR_DIR}/model/sft_model_step_22461.bin", map_location=device, weights_only=True)
)
model.to(device)
model.eval()


NsModel(
  (emb): Embedding(32002, 768)
  (layers): ModuleList(
    (0-11): 12 x TransformerBlock(
      (rms_norm1): RMSNorm()
      (attn): GQAAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=False)
        (w_k): Linear(in_features=768, out_features=128, bias=False)
        (w_v): Linear(in_features=768, out_features=128, bias=False)
        (w_o): Linear(in_features=768, out_features=768, bias=False)
      )
      (rms_norm2): RMSNorm()
      (ffn): FeedForward(
        (fc1): Linear(in_features=768, out_features=2048, bias=False)
        (fc2): Linear(in_features=768, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=768, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_out): Linear(in_features=768, out_features=32002, bias=False)
)

In [6]:
# inference with attention mask

prompts = [
    apply_chat_template("tell me a joke"),
    apply_chat_template("write a python sort function")
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)

In [7]:
input_tensor 

tensor([[32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 30370,   482,   261, 11202, 32001,
           224,   202,     1,     1],
        [32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 18617,   261, 14079, 13253,  3486,
          3054, 32001,   224,   202]], device='cuda:1')

In [8]:
attn_mask_tensor

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:1')

In [9]:
# no kv-cache + atten_mask

max_output = 500

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_tensor, attention_mask=attn_mask_tensor)
        # logits = model.forward(input_tensor, attention_mask=None)

        logits = logits[:, -1, :]

        # 概率采样
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )
        
        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 system
You are a helpful assistant. 
 user
tell me a joke 
ner, coz I'm a parrot, but I use a lot of alchemy to extract information from outclass mice, and add it into my image using a fraud scanner.

**Encryption:**
Had you ever set up an encryption algorithm against a specific loss function, like a linear programming line? Then, let's focus on a little derivation to get you started.

**Encryption:**
With it, you can extract information in various ways, like:

1. **Output:** Since variables cannot be processed, you can't make data texture copies, but you can string them to the output of the hidden layer.
2. **Data compression:** The output of the hidden layer represents the input sequence, rather than the binary code.
3. **Layout:** The output of the hidden layer is fragmented, meaning it's only accessible to the left of the hidden layer.

**Advantages:**
Just as encryption in the hidden layer is a bit more "echo-dependent," it offers the necessary functionality, allowing you to prot

In [10]:
# inference with attention mask

prompts = [
    apply_chat_template("tell me a joke"),
    apply_chat_template("write a python sort function")
]

pad_token_id = tokenizer.token_to_id("<pad>")

input_ids = tokenizer.encode_batch(prompts, add_special_tokens=False)

max_len = max(len(ids) for ids in input_ids)

input_list = []
mask_list = []

for item in input_ids:
    ids = item.ids
    pad_len = max_len - len(ids)

    if pad_len > 0:
        pad_ids = torch.tensor(ids + [pad_token_id] * pad_len, dtype=torch.long)
        attn_mask = torch.tensor([1] * len(ids) + [0] * pad_len, dtype=torch.long)
        
    else:
        pad_ids = torch.tensor(ids , dtype=torch.long)
        attn_mask = torch.tensor([1] * max_len, dtype=torch.long)

    input_list.append(pad_ids)
    mask_list.append(attn_mask)

input_tensor = torch.vstack(input_list).to(device)
attn_mask_tensor = torch.vstack(mask_list).to(device)

In [11]:
input_tensor

tensor([[32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 30370,   482,   261, 11202, 32001,
           224,   202,     1,     1],
        [32000,  1036,   202,  1684,   377,   261,  5856,  7475,    17, 32001,
           224,   202, 32000,  3164,   202, 18617,   261, 14079, 13253,  3486,
          3054, 32001,   224,   202]], device='cuda:1')

In [12]:
# kv-cache + atten_mask

max_output = 200


with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_tensor, use_cache=True, attention_mask=attn_mask_tensor)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        attn_mask_tensor = torch.cat([attn_mask_tensor, torch.ones((attn_mask_tensor.shape[0], 1), device=attn_mask_tensor.device, dtype=attn_mask_tensor.dtype)], dim=1)
        # NOTE: 使用kvcache时，只传入new token
        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True, attention_mask=attn_mask_tensor)

outputs = input_tensor.cpu().numpy()

for output in outputs:
    print(tokenizer.decode(output))
    print("----------------------------")

 system
You are a helpful assistant. 
 user
tell me a joke 
musica: 😇!

Bored with the concept of "muscle-humping" or "destination: mindless entertainment." I'm rambling on about the idea that there are no shortcuts to a career like this. While these were an obvious starting point, I hope they make the puzzle anyhow. Here's a humorous excerpt:

**The concept of "treasures"**: Who didn't invent this equation in college? That's where the unfortunate fact came in. Developing a costume that allows guests to freely be their own conscience, surrounded by strangers, and even suddenly associate with their sexual impersonation. This concept was first introduced by hipsters, while embracing atrocities like the Great Gatsby murder.

**Identity theft and violence**: As a journalist, I found myself curating a massive social media network consisting of strangers, users, and creators. The idea was that all humans were born with navel
----------------------------
 system
You are a helpful assistant. 
